In [1]:
! pip install "mcp[cli]" ollama nest_asyncio numpy

In [2]:
import nest_asyncio
nest_asyncio.apply()

import re
import numpy as np
import ollama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

CHAT_MODEL  = "gemma4:31b-cloud"
EMBED_MODEL = "embeddinggemma:latest"

SERVER = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-everything"]
)

print("config ready")

config ready


## FETCH EVERYTHING FROM THE MCP SERVER"

In [3]:
async def ingest_from_mcp(max_resources=25):
    mcp_tools, resources, prompts = [], [], []
    errlog = open("server_errors.log", "w")

    async with stdio_client(SERVER, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for t in (await session.list_tools()).tools:
                mcp_tools.append({"name": t.name,
                                  "description": t.description or "",
                                  "input_schema": t.input_schema})

            for r in (await session.list_resources()).resources[:max_resources]:
                try:
                    c = await session.read_resource(r.uri)
                    text = "".join(getattr(x, "text", "") or "" for x in c.contents)
                    if text.strip():
                        resources.append({"id": str(r.uri), "text": text})
                except Exception:
                    pass

            for p in (await session.list_prompts()).prompts:
                text = f"{p.name}: {p.description or ''}"
                try:
                    got = await session.get_prompt(p.name, {})
                    for m in got.messages:
                        text += "\n" + (getattr(m.content, "text", "") or "")
                except Exception:
                    pass
                prompts.append({"id": p.name, "text": text})

    errlog.close()
    return mcp_tools, resources, prompts

MCP_TOOLS, resources_data, prompts_data = await ingest_from_mcp()
MCP_TOOLS_BY_NAME = {t["name"]: t for t in MCP_TOOLS}
print(f"tools: {len(MCP_TOOLS)} | resources: {len(resources_data)} | prompts: {len(prompts_data)}")

tools: 13 | resources: 7 | prompts: 4


In [23]:
# NOW EVERYTHING IS WITH THE AI APPLICATION , BEFORE SENDING IT TO LLM WE HAVE TO DO ENGINEERRING
MCP_TOOLS

[{'name': 'echo',
  'description': 'Echoes back the input string',
  'input_schema': {'$schema': 'http://json-schema.org/draft-07/schema#',
   'type': 'object',
   'properties': {'message': {'type': 'string',
     'description': 'Message to echo'}},
   'required': ['message']}},
 {'name': 'get-annotated-message',
  'description': 'Demonstrates how annotations can be used to provide metadata about content.',
  'input_schema': {'$schema': 'http://json-schema.org/draft-07/schema#',
   'type': 'object',
   'properties': {'messageType': {'type': 'string',
     'enum': ['error', 'success', 'debug'],
     'description': 'Type of message to demonstrate different annotation patterns'},
    'includeImage': {'default': False,
     'description': 'Whether to include an example image',
     'type': 'boolean'}},
   'required': ['messageType']}},
 {'name': 'get-env',
  'description': 'Returns all environment variables, helpful for debugging MCP server configuration',
  'input_schema': {'$schema': 'ht

## BUILDING HELPER FUNCTIONS

In [7]:
def get_embedding(text, task_type="search_document"):
    response = ollama.embed(model=EMBED_MODEL, input=f"{task_type}: {text}")
    return response["embeddings"][0]

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def clean_docstring(doc):
    if not doc:
        return ""
    return re.split(r"\n\s*(Args:|Returns:)", doc)[0].strip()

print("helpers ready")

helpers ready


### BUILDING TOOL REGISTRY

In [8]:
TOOL_EMBEDDINGS = {}
for t in MCP_TOOLS:
    text = t["name"] + " " + clean_docstring(t["description"])
    TOOL_EMBEDDINGS[t["name"]] = get_embedding(text, task_type="search_document")

print(f"Precomputed embeddings for {len(TOOL_EMBEDDINGS)} tools")

Precomputed embeddings for 13 tools


### SEARCH TOOL [DEFERRED LOADING]

In [10]:
def search_tools(query, max_results=3):
    query_emb   = get_embedding(query, task_type="search_query")
    query_words = set(query.lower().split())

    scored = []
    for t in MCP_TOOLS:
        name = t["name"]
        text = (name + " " + clean_docstring(t["description"])).lower()
        tool_words = set(text.split())

        keyword_score   = len(query_words & tool_words) / max(len(query_words), 1)
        embedding_score = cosine_similarity(query_emb, TOOL_EMBEDDINGS[name])
        combined        = (0.6 * keyword_score) + (0.4 * embedding_score)
        scored.append((combined, name))

    scored.sort(reverse=True)
    return [name for score, name in scored[:max_results]]

print("search_tools ready")

search_tools ready


In [11]:
# SEARCH TOOL SCHEMA 
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for tools that are not yet loaded into this conversation, "
            "by meaning as well as keywords. Use this the moment you realize "
            "you need a capability you don't currently have.\n\n"
            "Returns matching tool names, ranked by relevance. Once a tool is "
            "returned here, it becomes directly callable on your next turn - "
            "you do not need to search for it again later.\n\n"
            "If the top result doesn't look right, try rephrasing your query "
            "rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Describe the capability you need in a natural phrase - "
                        "full context helps more than a few bare keywords."
                    ),
                },
                "max_results": {
                    "type": "integer",
                    "description": "Max tools to return. Defaults to 3.",
                },
            },
            "required": ["query"],
        },
    },
}
print("SEARCH_TOOL ready")

SEARCH_TOOL ready


### KNOWLEDGE SOURCE FOR RAG

In [12]:
# KNOWLEDGE (RAG) from resources + prompts — a simple in-memory list, precomputed once
knowledge = []
for r in resources_data:
    knowledge.append({"source": "resource", "id": r["id"], "text": r["text"],
                      "vec": get_embedding(r["text"], task_type="search_document")})
for p in prompts_data:
    knowledge.append({"source": "prompt", "id": p["id"], "text": p["text"],
                      "vec": get_embedding(p["text"], task_type="search_document")})

# (optional) add real policy docs so RAG is meaningful for the demo
for doc_id, text in [
    ("policy:refund",   "DataFlow refund policy: items may be returned within 30 days of "
                        "delivery for a full refund if unused and in original packaging. "
                        "Refunds process within 5 business days."),
    ("policy:shipping", "DataFlow shipping: standard takes 3-5 business days. Express takes "
                        "1-2 business days for an extra $12. We do not ship to PO boxes."),
]:
    knowledge.append({"source": "doc", "id": doc_id, "text": text,
                      "vec": get_embedding(text, task_type="search_document")})

print(f"knowledge entries: {len(knowledge)}")

knowledge entries: 13


In [15]:
# knowledge

### KNOWLEDGE RETIREVAL

In [16]:
KNOWLEDGE_THRESHOLD = 0.45   # inject knowledge only if a chunk scores >= this

def search_knowledge(query, threshold=KNOWLEDGE_THRESHOLD):
    qv = get_embedding(query, task_type="search_query")
    scored = sorted(
        ((cosine_similarity(qv, e["vec"]), e) for e in knowledge),
        key=lambda x: x[0], reverse=True
    )
    return [(s, e) for s, e in scored if s >= threshold]

print("search_knowledge ready")

search_knowledge ready


In [17]:
search_knowledge("Can I return an item I bought two weeks ago?")

[(0.5478353275433956,
  {'source': 'doc',
   'id': 'policy:refund',
   'text': 'DataFlow refund policy: items may be returned within 30 days of delivery for a full refund if unused and in original packaging. Refunds process within 5 business days.',
   'vec': [-0.04935368,
    0.0012895114,
    0.05315619,
    0.02235968,
    -0.012639418,
    0.05048632,
    -0.086089544,
    0.0028365622,
    0.04825566,
    0.016164538,
    0.008367498,
    -0.00015800756,
    0.037516754,
    -0.00243538,
    0.027697586,
    0.030725854,
    -0.014616353,
    0.05332247,
    0.00232258,
    -0.028964475,
    0.050957352,
    -0.045763493,
    -0.01549718,
    0.0013009659,
    -0.040748302,
    0.018344559,
    -0.00521545,
    0.0067491825,
    0.023416026,
    0.0068177227,
    0.022308873,
    -0.0020055168,
    -0.00078728254,
    -0.015681783,
    0.032160025,
    -0.013405623,
    0.0025915487,
    0.009980584,
    0.085113995,
    -0.030352859,
    0.033499125,
    0.021737905,
    -0.01487

#### CREATING SYSTEM PROMPT AND CREATING FUNCTIONS IN OLLAMA FORMAT

In [18]:
SYSTEM_PROMPT = """<role>
You are DataFlow Assistant — a helpful, accurate customer-support agent.
You help users with DataFlow orders, refunds, shipping, and account questions.
Answer only from the information available to you. Never guess or invent facts.
</role>

<instructions>
For every user message, decide which of the three cases below applies, then act.

## Case 1 — Knowledge / Policy question
- If a `<reference_information>` block is provided in the conversation, answer using it.
- Do not add any fact that is not in that block.

## Case 2 — Action or live data (look something up, run something, fetch real values)
- You start with NO action tools loaded — only `search_tools` is available.
- **Step 1:** call `search_tools` to find the capability you need.
- **Step 2:** on your next turn the matching tool becomes callable — call it.
- Never perform an action from memory. If it needs a tool, use a tool.

## Case 3 — General conversation (greetings, small talk, simple replies)
- Answer directly. Do not search for tools or reference information.
</instructions>

<rules>
- Only call tools that are currently available to you.
- NEVER invent, guess, or assume a tool name — if you don't have it, search for it first.
- Be concise and accurate.
- Do not make up facts, numbers, policies, or results.
</rules>"""

In [19]:
def ollama_def(name):
    t = MCP_TOOLS_BY_NAME[name]
    return {"type": "function",
            "function": {"name": t["name"],
                         "description": t["description"],
                         "parameters": t["input_schema"]}}


### CREATING AGENTIC LOOP

In [20]:
async def run_agent(user_query, model=CHAT_MODEL, max_steps=6, verbose=True):

    # ---- RAG: retrieve relevant knowledge --------------------------------
    hits = search_knowledge(user_query)
    if verbose:
        print(f"📚 RAG fired (top {hits[0][0]:.2f}): {[e['id'] for _, e in hits]}"
              if hits else "📚 RAG skipped (nothing relevant)")

    # ---- build the user turn: reference travels WITH the query -----------
    if hits:
        block = "\n\n".join(f"[{e['source']}] {e['text']}" for _, e in hits)
        user_content = f"<reference_information>\n{block}\n</reference_information>\n\n{user_query}"
    else:
        user_content = user_query



  # DEFERRED LOADING TOOL CALLING.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    available = [SEARCH_TOOL]        # single source of truth for what's callable
    tools_called = []

    errlog = open("server_errors.log", "w")
    async with stdio_client(SERVER, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for step in range(max_steps):
                resp = ollama.chat(model=model, messages=messages, tools=available)
                msg = resp.message
                messages.append(msg)

                if verbose:
                    print(f"   🧮 step {step+1}: context = {resp.get('prompt_eval_count')} tokens "
                          f"| tools available = {len(available)}")

                if not msg.tool_calls:   # IF NO TOOL CALL GIVE THE OUTPUT 
                    if verbose:
                        print(f"✅ done in {step+1} step(s) | tools called: {tools_called or 'none'}\n")
                    errlog.close()
                    return msg.content

                available_names = {t["function"]["name"] for t in available}

                for tc in msg.tool_calls: # IF TOOL CALL , COLLECTING THE NAME OF FUNCTION AND ITS ARGUMENT TO BE CALLED.
                    name, args = tc.function.name, tc.function.arguments
                    tools_called.append(name)

                    if name == "search_tools":  # IF TOOL IS SEARCH TOOL
                        found = search_tools(args["query"], args.get("max_results", 3))
                        for n in found:
                            if n not in available_names:
                                available.append(ollama_def(n))
                                available_names.add(n)
                        out = f"Loaded (now callable): {', '.join(found)}" if found else "No tools matched."
                        if verbose:
                            print(f"   🔍 search_tools('{args['query']}') -> {found}")

                    elif name in available_names:
                        try:
                            res = await session.call_tool(name, args)
                            out = res.content[0].text if res.content else ""
                        except Exception as e:
                            out = f"ERROR: {e}"
                        if verbose:
                            print(f"   → {name}({args})")

                    else:
                        out = f"ERROR: '{name}' is not loaded. Use search_tools first."
                        if verbose:
                            print(f"   ⚠️ blocked invented tool: {name}")

                    messages.append({"role": "tool", "content": out})

    errlog.close()
    return "Stopped: reached max steps."

print("agent ready")

agent ready


In [21]:
print("Q1:", await run_agent("Can I return an item I bought two weeks ago?"));          print("="*70)
print("Q3:", await run_agent("Hi, how are you today?"));                                print("="*70)

📚 RAG fired (top 0.55): ['policy:refund']
   🧮 step 1: context = 589 tokens | tools available = 1
✅ done in 1 step(s) | tools called: none

Q1: Yes, you can return an item bought two weeks ago, provided it is unused and in its original packaging. Items may be returned within 30 days of delivery for a full refund.
📚 RAG skipped (nothing relevant)
   🧮 step 1: context = 534 tokens | tools available = 1
✅ done in 1 step(s) | tools called: none

Q3: I'm doing well, thank you for asking! How can I help you with your DataFlow account or orders today?


In [22]:
print("A3:", await run_agent("Run a long background operation and tell me when it completes and give me output of it too")); print("="*70)

📚 RAG fired (top 0.45): ['args-prompt']
   🧮 step 1: context = 576 tokens | tools available = 1
   🔍 search_tools('run a long background operation and get the output when it completes') -> ['trigger-long-running-operation', 'gzip-file-as-resource', 'simulate-research-query']
   🧮 step 2: context = 1011 tokens | tools available = 4
   → trigger-long-running-operation({'duration': 10, 'steps': 5})
   🧮 step 3: context = 1060 tokens | tools available = 4
✅ done in 3 step(s) | tools called: ['search_tools', 'trigger-long-running-operation']

A3: The long background operation has completed. It ran for 10 seconds across 5 steps.


In [24]:
print("Q1:", await run_agent("Does your shipping policy allow delivery to a PO box?")); print("="*70)

📚 RAG fired (top 0.48): ['policy:shipping']
   🧮 step 1: context = 590 tokens | tools available = 1
✅ done in 1 step(s) | tools called: none

Q1: No, we do not ship to PO boxes.


In [25]:
print("R1:", await run_agent("How does this system actually work internally?")); print("="*70)

📚 RAG fired (top 0.47): ['demo://resource/static/document/how-it-works.md']
   🧮 step 1: context = 1197 tokens | tools available = 1
✅ done in 1 step(s) | tools called: none

R1: Internally, the system operates through several specialized modules:

*   **Tool Registration:** Most tools are registered during the Server Factory execution. However, certain tools that require specific client capabilities (such as `get-roots-list`, `trigger-elicitation-request`, and `trigger-sampling-request`) are registered conditionally via the `registerConditionalTools(server)` function, which is triggered by an `oninitialized` handler after the client handshake is complete.
*   **Resource Management:** 
    *   **Subscriptions:** The system uses a map (`Map<uri, Set<sessionId>>`) to track subscribers per URI. Handlers are installed via `setSubscriptionHandlers(server)`, and updates are managed by the `toggle-subscriber-updates` tool.
    *   **Session-scoped Resources:** The system can create temporary,